In [14]:
import pickle
from pathlib import Path

import pandas as pd
import numpy as np

OUTPUT_DIR = Path("output")

# 1) загрузка всех batch-файлов
files = sorted(OUTPUT_DIR.glob("spreads.pkl"))
print(f"Found files: {len(files)}")
for f in files[:10]:
    print(f)

records = []

for path in files:
    with path.open("rb") as fh:
        while True:
            try:
                batch = pickle.load(fh)
            except EOFError:
                break

            if isinstance(batch, list):
                records.extend(batch)
            else:
                print(f"Skip non-list payload in {path}: type={type(batch)}")

df = pd.DataFrame(records)
print("\nRaw shape:", df.shape)

if df.empty:
    raise ValueError("No records loaded")

# 2) приведение числовых полей
num_cols = [
    "spread_long",
    "spread_short",
    "okx_latency_ms",
    "bybit_latency_ms",
    "calc_local_ts_ms",
    "okx_local_recv_ts_ms",
    "okx_ts_ms",
    "bybit_local_recv_ts_ms",
    "bybit_ts_ms",
]
for col in num_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

# 3) datetime-поля
df["calc_dt"] = pd.to_datetime(df["calc_local_ts_ms"], unit="ms", errors="coerce")
df["okx_local_recv_dt"] = pd.to_datetime(df["okx_local_recv_ts_ms"], unit="ms", errors="coerce")
df["bybit_local_recv_dt"] = pd.to_datetime(df["bybit_local_recv_ts_ms"], unit="ms", errors="coerce")

# 4) freshness на момент расчёта спреда
df["okx_freshness_ms"] = df["calc_local_ts_ms"] - df["okx_local_recv_ts_ms"]
df["bybit_freshness_ms"] = df["calc_local_ts_ms"] - df["bybit_local_recv_ts_ms"]

# 5) event time = локальное время прихода триггерного сообщения
# если trigger == okx, индексируем по okx_local_recv_ts_ms
# если trigger == bybit, индексируем по bybit_local_recv_ts_ms
df["event_local_ts_ms"] = np.where(
    df["trigger"].eq("okx"),
    df["okx_local_recv_ts_ms"],
    df["bybit_local_recv_ts_ms"],
)

df["event_dt"] = pd.to_datetime(df["event_local_ts_ms"], unit="ms", errors="coerce")

# 6) сортировка и временной индекс
df = df.sort_values(["event_dt", "base_coin", "trigger"]).set_index("event_dt")

# 7) полезный порядок колонок
front_cols = [
    "base_coin",
    "trigger",
    "spread_long",
    "spread_short",
    "okx_latency_ms",
    "bybit_latency_ms",
    "okx_freshness_ms",
    "bybit_freshness_ms",
    "calc_dt",
    "okx_local_recv_dt",
    "bybit_local_recv_dt",
    "calc_local_ts_ms",
    "event_local_ts_ms",
]
rest_cols = [c for c in df.columns if c not in front_cols]
df = df[front_cols + rest_cols]

print("\nTime-indexed shape:", df.shape)
print("\nIndex range:")
print(df.index.min(), "->", df.index.max())

print("\nHead:")
display(df.head(10))

print("\nTail:")
display(df.tail(10))

print("\nLatency summary:")
display(
    df[["okx_latency_ms", "bybit_latency_ms", "okx_freshness_ms", "bybit_freshness_ms"]]
    .describe(percentiles=[0.5, 0.9, 0.95, 0.99])
)

print("\nCounts by base_coin:")
display(df["base_coin"].value_counts().head(20))

print("\nCounts by base_coin + trigger:")
display(df.groupby(["base_coin", "trigger"]).size().sort_values(ascending=False).head(30))

print("\nWorst bybit latency rows:")
display(
    df.sort_values("bybit_latency_ms", ascending=False)[
        [
            "base_coin",
            "trigger",
            "spread_long",
            "spread_short",
            "okx_latency_ms",
            "bybit_latency_ms",
            "okx_freshness_ms",
            "bybit_freshness_ms",
            "calc_dt",
            "okx_local_recv_dt",
            "bybit_local_recv_dt",
        ]
    ].head(20)
)

print("\nWorst freshness rows:")
display(
    df.assign(max_freshness_ms=df[["okx_freshness_ms", "bybit_freshness_ms"]].max(axis=1))
      .sort_values("max_freshness_ms", ascending=False)[
          [
              "base_coin",
              "trigger",
              "spread_long",
              "spread_short",
              "okx_latency_ms",
              "bybit_latency_ms",
              "okx_freshness_ms",
              "bybit_freshness_ms",
              "calc_dt",
          ]
      ].head(20)
)

Found files: 1
output/spreads.pkl

Raw shape: (46150180, 11)

Time-indexed shape: (46150180, 17)

Index range:
2026-07-16 21:08:09.128992920 -> 2026-07-17 08:46:59.429110107

Head:


,base_coin,trigger,spread_long,spread_short,okx_latency_ms,bybit_latency_ms,okx_freshness_ms,bybit_freshness_ms,calc_dt,okx_local_recv_dt,bybit_local_recv_dt,calc_local_ts_ms,event_local_ts_ms,okx_local_recv_ts_ms,okx_ts_ms,bybit_local_recv_ts_ms,bybit_ts_ms
event_dt,,,,,,,,,,,,,,,,,
2026-07-16 21:08:09.128992920,BSP,okx,-0.062383,-0.374883,121.992920,2562.170166,0.021973,226.844727,2026-07-16 21:08:09.129014893,2026-07-16 21:08:09.128992920,2026-07-16 21:08:08.902170166,1.784236e+12,1.784236e+12,1.784236e+12,1.784236e+12,1.784236e+12,1.784236e+12
2026-07-16 21:08:09.130397949,IREN,okx,-0.113766,-0.113766,123.397949,1659.851074,0.018311,225.565186,2026-07-16 21:08:09.130416260,2026-07-16 21:08:09.130397949,2026-07-16 21:08:08.904851074,1.784236e+12,1.784236e+12,1.784236e+12,1.784236e+12,1.784236e+12,1.784236e+12
2026-07-16 21:08:09.133250000,SMCI,okx,-0.201369,-0.040274,129.250000,2661.994873,0.019043,231.274170,2026-07-16 21:08:09.133269043,2026-07-16 21:08:09.133250000,2026-07-16 21:08:08.901994873,1.784236e+12,1.784236e+12,1.784236e+12,1.784236e+12,1.784236e+12,1.784236e+12
2026-07-16 21:08:09.340291992,SMCI,bybit,-0.201369,-0.040274,129.250000,101.291992,207.079834,0.037842,2026-07-16 21:08:09.340329834,2026-07-16 21:08:09.133250000,2026-07-16 21:08:09.340291992,1.784236e+12,1.784236e+12,1.784236e+12,1.784236e+12,1.784236e+12,1.784236e+12
2026-07-16 21:08:09.420442139,ANIME,okx,0.000000,-0.073421,117.442139,281.813965,0.038818,521.666992,2026-07-16 21:08:09.420480957,2026-07-16 21:08:09.420442139,2026-07-16 21:08:08.898813965,1.784236e+12,1.784236e+12,1.784236e+12,1.784236e+12,1.784236e+12,1.784236e+12
2026-07-16 21:08:09.426305176,AR,okx,0.015134,-0.070671,122.305176,296.338135,0.043945,528.010986,2026-07-16 21:08:09.426349121,2026-07-16 21:08:09.426305176,2026-07-16 21:08:08.898338135,1.784236e+12,1.784236e+12,1.784236e+12,1.784236e+12,1.784236e+12,1.784236e+12
2026-07-16 21:08:09.428218750,RVN,okx,0.104167,-0.182529,125.218750,317.632080,0.038330,551.625000,2026-07-16 21:08:09.428257079,2026-07-16 21:08:09.428218750,2026-07-16 21:08:08.876632080,1.784236e+12,1.784236e+12,1.784236e+12,1.784236e+12,1.784236e+12,1.784236e+12
2026-07-16 21:08:09.428277100,MON,okx,0.013900,-0.064905,125.277100,2186.127930,0.022949,540.172119,2026-07-16 21:08:09.428300049,2026-07-16 21:08:09.428277100,2026-07-16 21:08:08.888127929,1.784236e+12,1.784236e+12,1.784236e+12,1.784236e+12,1.784236e+12,1.784236e+12
2026-07-16 21:08:09.428316895,TRX,okx,-0.015486,0.009291,124.316895,299.991943,0.025146,528.350098,2026-07-16 21:08:09.428342041,2026-07-16 21:08:09.428316895,2026-07-16 21:08:08.899991943,1.784236e+12,1.784236e+12,1.784236e+12,1.784236e+12,1.784236e+12,1.784236e+12



Tail:


,base_coin,trigger,spread_long,spread_short,okx_latency_ms,bybit_latency_ms,okx_freshness_ms,bybit_freshness_ms,calc_dt,okx_local_recv_dt,bybit_local_recv_dt,calc_local_ts_ms,event_local_ts_ms,okx_local_recv_ts_ms,okx_ts_ms,bybit_local_recv_ts_ms,bybit_ts_ms
event_dt,,,,,,,,,,,,,,,,,
2026-07-17 08:46:59.363730957,AVNT,bybit,-0.044563,-0.055692,0.162842,-36.269043,157.582031,0.013916,2026-07-17 08:46:59.363744873,2026-07-17 08:46:59.206162842,2026-07-17 08:46:59.363730957,1.784278e+12,1.784278e+12,1.784278e+12,1.784278e+12,1.784278e+12,1.784278e+12
2026-07-17 08:46:59.367650879,LRC,bybit,2.357414,-2.968750,3.876221,-33.349121,63.791748,0.017090,2026-07-17 08:46:59.367667969,2026-07-17 08:46:59.303876221,2026-07-17 08:46:59.367650879,1.784278e+12,1.784278e+12,1.784278e+12,1.784278e+12,1.784278e+12,1.784278e+12
2026-07-17 08:46:59.368274902,WOO,bybit,-0.202259,0.050505,547.589111,-33.725098,3314.699951,0.014160,2026-07-17 08:46:59.368289062,2026-07-17 08:46:56.053589111,2026-07-17 08:46:59.368274902,1.784278e+12,1.784278e+12,1.784278e+12,1.784278e+12,1.784278e+12,1.784278e+12
2026-07-17 08:46:59.369147949,IWM,bybit,0.363426,-0.450235,3.577148,-33.852051,63.584961,0.014160,2026-07-17 08:46:59.369162109,2026-07-17 08:46:59.305577148,2026-07-17 08:46:59.369147949,1.784278e+12,1.784278e+12,1.784278e+12,1.784278e+12,1.784278e+12,1.784278e+12
2026-07-17 08:46:59.392015869,DRAM,bybit,-0.040560,0.000000,3.283936,-11.984131,86.764160,0.032227,2026-07-17 08:46:59.392048096,2026-07-17 08:46:59.305283936,2026-07-17 08:46:59.392015869,1.784278e+12,1.784278e+12,1.784278e+12,1.784278e+12,1.784278e+12,1.784278e+12
2026-07-17 08:46:59.422510986,ETH,bybit,-0.019144,0.018047,5.520020,14.510986,117.010010,0.019043,2026-07-17 08:46:59.422530029,2026-07-17 08:46:59.305520020,2026-07-17 08:46:59.422510986,1.784278e+12,1.784278e+12,1.784278e+12,1.784278e+12,1.784278e+12,1.784278e+12
2026-07-17 08:46:59.422535889,ETH,bybit,-0.019144,0.018047,5.520020,4.535889,117.021973,0.006104,2026-07-17 08:46:59.422541992,2026-07-17 08:46:59.305520020,2026-07-17 08:46:59.422535889,1.784278e+12,1.784278e+12,1.784278e+12,1.784278e+12,1.784278e+12,1.784278e+12
2026-07-17 08:46:59.422550049,BE,bybit,-0.066151,-0.193493,2.046143,14.550049,618.509766,0.005859,2026-07-17 08:46:59.422555908,2026-07-17 08:46:58.804046143,2026-07-17 08:46:59.422550049,1.784278e+12,1.784278e+12,1.784278e+12,1.784278e+12,1.784278e+12,1.784278e+12
2026-07-17 08:46:59.422562988,SKY,bybit,-0.033630,-0.050454,202.820801,12.562988,1017.749268,0.007080,2026-07-17 08:46:59.422570068,2026-07-17 08:46:58.404820801,2026-07-17 08:46:59.422562988,1.784278e+12,1.784278e+12,1.784278e+12,1.784278e+12,1.784278e+12,1.784278e+12



Latency summary:


...

base_coin  trigger
BTC        bybit      564312
ETH        bybit      512841
SNDK       bybit      438899
HYPE       bybit      365152
SOXL       bybit      343508
MU         bybit      341286
SKHYNIX    bybit      338795
XRP        bybit      324967
SOL        bybit      317501
TRB        bybit      301159
WDC        bybit      298346
LIT        bybit      295759
LAB        bybit      279947
DOGE       bybit      277925
ADA        bybit      266316
LITE       bybit      261000
SNDK       okx        253326
ETH        okx        252528
ARM        bybit      244837
SOL        okx        243074
HOME       bybit      239457
IWM        bybit      237686
SOXL       okx        237428
BTC        okx        234338
MU         okx        232002
ADA        okx        213134
SKHYNIX    okx        211512
BE         bybit      208757
DOGE       okx        205995
HYPE       okx        204751
dtype: int64


Worst bybit latency rows:


,base_coin,trigger,spread_long,spread_short,okx_latency_ms,bybit_latency_ms,okx_freshness_ms,bybit_freshness_ms,calc_dt,okx_local_recv_dt,bybit_local_recv_dt
event_dt,,,,,,,,,,,
2026-07-16 23:30:16.054895996,LRC,bybit,-0.081633,-0.163265,83.883057,20454.895996,862.037842,0.024902,2026-07-16 23:30:16.054920898,2026-07-16 23:30:15.192883057,2026-07-16 23:30:16.054895996
2026-07-16 23:30:16.054928711,LRC,bybit,-0.081633,-0.163265,83.883057,20354.928711,862.053955,0.008301,2026-07-16 23:30:16.054937012,2026-07-16 23:30:15.192883057,2026-07-16 23:30:16.054928711
2026-07-16 23:30:15.988296875,BZ,bybit,-0.023915,0.000000,91.333984,18344.296875,287.979248,0.016357,2026-07-16 23:30:15.988313232,2026-07-16 23:30:15.700333984,2026-07-16 23:30:15.988296875
2026-07-16 23:30:16.078182861,LRC,bybit,-0.081633,-0.163265,83.883057,18178.182861,885.320801,0.020996,2026-07-16 23:30:16.078203857,2026-07-16 23:30:15.192883057,2026-07-16 23:30:16.078182861
2026-07-16 23:30:16.459791260,BILL,bybit,0.074377,-0.148920,86.478027,18084.791260,367.335205,0.021973,2026-07-16 23:30:16.459813232,2026-07-16 23:30:16.092478027,2026-07-16 23:30:16.459791260
2026-07-16 23:30:16.080300049,LRC,bybit,-0.081633,-0.163265,83.883057,18080.300049,887.431152,0.014160,2026-07-16 23:30:16.080314209,2026-07-16 23:30:15.192883057,2026-07-16 23:30:16.080300049
2026-07-16 23:30:16.479427979,BILL,bybit,0.074377,-0.148920,86.478027,17933.427979,386.964844,0.014893,2026-07-16 23:30:16.479442871,2026-07-16 23:30:16.092478027,2026-07-16 23:30:16.479427979
2026-07-16 23:30:16.479447021,BILL,bybit,0.074377,-0.148920,86.478027,17904.447021,386.977051,0.008057,2026-07-16 23:30:16.479455078,2026-07-16 23:30:16.092478027,2026-07-16 23:30:16.479447021
2026-07-16 23:30:16.501388184,BILL,bybit,0.074377,-0.148920,86.478027,17876.388184,408.930908,0.020752,2026-07-16 23:30:16.501408936,2026-07-16 23:30:16.092478027,2026-07-16 23:30:16.501388184



Worst freshness rows:


,base_coin,trigger,spread_long,spread_short,okx_latency_ms,bybit_latency_ms,okx_freshness_ms,bybit_freshness_ms,calc_dt
event_dt,,,,,,,,,
2026-07-17 08:46:54.104938965,BSP,bybit,0.409449,-0.949367,96.698975,-35.061035,1.241221e+07,0.010010,2026-07-17 08:46:54.104948975
2026-07-17 08:46:53.830954102,BSP,bybit,0.409449,-0.949367,96.698975,3780.954102,1.241193e+07,0.018799,2026-07-17 08:46:53.830972900
2026-07-17 08:46:58.160142090,XLE,bybit,0.139592,-0.402591,97.856201,126.142090,1.241066e+07,0.007080,2026-07-17 08:46:58.160149170
2026-07-17 08:46:55.039057861,XLE,bybit,0.139592,-0.402591,97.856201,5.057861,1.240754e+07,0.007324,2026-07-17 08:46:55.039065185
2026-07-17 08:46:53.831211182,XLE,bybit,0.139592,-0.402591,97.856201,1797.211182,1.240633e+07,0.038818,2026-07-17 08:46:53.831250000
2026-07-17 08:46:56.355992920,ONDS,bybit,-2.175614,1.900674,102.597900,134.992920,1.240575e+07,0.002930,2026-07-17 08:46:56.355995850
2026-07-17 08:46:56.355989014,ONDS,bybit,-2.175614,1.900674,102.597900,144.989014,1.240575e+07,0.003174,2026-07-17 08:46:56.355992188
2026-07-17 08:46:56.355983887,ONDS,bybit,-2.175614,1.900674,102.597900,244.983887,1.240575e+07,0.002930,2026-07-17 08:46:56.355986816
2026-07-17 08:46:56.355979980,ONDS,bybit,-2.175614,1.916002,102.597900,254.979980,1.240575e+07,0.002930,2026-07-17 08:46:56.355982910


In [1]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.io as pio

pio.renderers.default = "browser"

plot_df = df.copy()

if not isinstance(plot_df.index, pd.DatetimeIndex):
    if "event_dt" in plot_df.columns:
        plot_df["event_dt"] = pd.to_datetime(plot_df["event_dt"], errors="coerce")
        plot_df = plot_df.set_index("event_dt")
    elif "calc_dt" in plot_df.columns:
        plot_df["calc_dt"] = pd.to_datetime(plot_df["calc_dt"], errors="coerce")
        plot_df = plot_df.set_index("calc_dt")
    else:
        raise ValueError("Нет временной оси: нужен DatetimeIndex или колонка event_dt/calc_dt")

plot_df = plot_df.sort_index()

SPREAD_COL = "spread_long"
TOP_N_COINS = 3
COIN_OFFSET = 30   
MAX_POINTS_PER_COIN = 100

coin_counts = plot_df["base_coin"].value_counts()
selected_coins = coin_counts.iloc[COIN_OFFSET:COIN_OFFSET + TOP_N_COINS].index.tolist()
legend_name_map = {
    coin: f"{COIN_OFFSET + i + 1}. {coin}"
    for i, coin in enumerate(selected_coins)
}
plot_df = plot_df[plot_df["base_coin"].isin(selected_coins)].copy()

top_coins = plot_df["base_coin"].value_counts().head(TOP_N_COINS).index.tolist()
plot_df = plot_df[plot_df["base_coin"].isin(top_coins)].copy()

coins = sorted(plot_df["base_coin"].unique())
coin_to_y = {coin: i for i, coin in enumerate(coins)}
plot_df["coin_y"] = plot_df["base_coin"].map(coin_to_y)

# время в секундах, но нормируем к старту окна, чтобы не было огромных unix-чисел
t0 = plot_df.index.min()
plot_df["time_s_from_start"] = (plot_df.index - t0).total_seconds()

# центрируем спред вокруг нуля для лучшей читаемости
zmin = plot_df[SPREAD_COL].min()
zmax = plot_df[SPREAD_COL].max()

fig = go.Figure()

for coin in coins:
    sub = plot_df[plot_df["base_coin"] == coin].sort_index().copy()

    if len(sub) > MAX_POINTS_PER_COIN:
        step = int(np.ceil(len(sub) / MAX_POINTS_PER_COIN))
        sub = sub.iloc[::step].copy()

    fig.add_trace(
        go.Scatter3d(
            x=sub["time_s_from_start"],
            y=sub["coin_y"],
            z=sub[SPREAD_COL],
            mode="lines+markers",
            name=legend_name_map[coin],
            line=dict(width=6),
            marker=dict(size=4),
            opacity=0.95,
            customdata=np.stack([
                sub.index.astype(str),
                sub["trigger"].astype(str),
                sub["okx_latency_ms"].fillna(np.nan),
                sub["bybit_latency_ms"].fillna(np.nan),
            ], axis=-1),
            hovertemplate=(
                "coin=%{text}<br>"
                "time=%{customdata[0]}<br>"
                "trigger=%{customdata[1]}<br>"
                f"{SPREAD_COL}=%{{z:.6f}}<br>"
                "okx_latency_ms=%{customdata[2]:.2f}<br>"
                "bybit_latency_ms=%{customdata[3]:.2f}<extra></extra>"
            ),
            text=[legend_name_map[coin]] * len(sub),
        )
    )

fig.update_layout(
    title=f"3D spread map: {SPREAD_COL} vs local event time vs coin",
    width=1500,
    height=950,
    scene=dict(
        xaxis=dict(
            title="Seconds from window start",
            backgroundcolor="rgb(245,245,245)",
            gridcolor="lightgray",
            zerolinecolor="gray",
        ),
        yaxis=dict(
            tickmode="array",
            backgroundcolor="rgb(245,245,245)",
            gridcolor="lightgray",
        ),
        zaxis=dict(
            title=SPREAD_COL,
            range=[zmin, zmax],
            backgroundcolor="rgb(245,245,245)",
            gridcolor="lightgray",
            zerolinecolor="black",
        ),
        aspectmode="manual",
        aspectratio=dict(x=2.8, y=1.5, z=1.2),
        camera=dict(
            eye=dict(x=1.9, y=1.6, z=1.1)
        ),
    ),
    legend=dict(font=dict(size=11)),
    margin=dict(l=0, r=0, t=50, b=0),
)

fig.show()

NameError: name 'df' is not defined

## Обработка pkl данных в parquet

In [ ]:



import pickle
from pathlib import Path

import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq


INPUT_PATH = Path("output/spreads_1.pkl")
PARQUET_DIR = Path("output/spreads_parquet_v2_1")
PARQUET_DIR.mkdir(parents=True, exist_ok=True)

NUM_COLS = [
    "spread_long",
    "spread_short",
    "okx_latency_ms",
    "bybit_latency_ms",
    "calc_local_ts_ms",
    "okx_local_recv_ts_ms",
    "okx_ts_ms",
    "bybit_local_recv_ts_ms",
    "bybit_ts_ms",
]


def normalize_batch(batch):
    df = pd.DataFrame(batch)
    if df.empty:
        return df

    for col in NUM_COLS:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")

    df["okx_freshness_ms"] = df["calc_local_ts_ms"] - df["okx_local_recv_ts_ms"]
    df["bybit_freshness_ms"] = df["calc_local_ts_ms"] - df["bybit_local_recv_ts_ms"]

    df["event_local_ts_ms"] = df["okx_local_recv_ts_ms"]
    mask_bybit = df["trigger"].eq("bybit")
    df.loc[mask_bybit, "event_local_ts_ms"] = df.loc[mask_bybit, "bybit_local_recv_ts_ms"]

    df["event_dt"] = pd.to_datetime(df["event_local_ts_ms"], unit="ms", errors="coerce")
    df = df[df["event_dt"].notna()].copy()

    df["event_date"] = df["event_dt"].dt.strftime("%Y-%m-%d")
    df["event_hour"] = df["event_dt"].dt.strftime("%H")
    df["max_freshness_ms"] = df[["okx_freshness_ms", "bybit_freshness_ms"]].max(axis=1)
    df["max_latency_ms"] = df[["okx_latency_ms", "bybit_latency_ms"]].max(axis=1)

    keep_cols = [
        "event_dt",
        "event_local_ts_ms",
        "event_date",
        "event_hour",
        "base_coin",
        "trigger",
        "spread_long",
        "spread_short",
        "okx_latency_ms",
        "bybit_latency_ms",
        "okx_freshness_ms",
        "bybit_freshness_ms",
        "max_freshness_ms",
        "max_latency_ms",
        "calc_local_ts_ms",
        "okx_local_recv_ts_ms",
        "okx_ts_ms",
        "bybit_local_recv_ts_ms",
        "bybit_ts_ms",
    ]
    keep_cols = [c for c in keep_cols if c in df.columns]
    return df[keep_cols].copy()


def write_partitioned_files(df: pd.DataFrame, batch_idx: int):
    if df.empty:
        return 0

    written = 0

    for (event_date, event_hour), sub in df.groupby(["event_date", "event_hour"], sort=False):
        part_dir = PARQUET_DIR / f"event_date={event_date}" / f"event_hour={event_hour}"
        part_dir.mkdir(parents=True, exist_ok=True)

        file_path = part_dir / f"batch_{batch_idx:06d}.parquet"
        table = pa.Table.from_pandas(sub.drop(columns=["event_date", "event_hour"]), preserve_index=False)
        pq.write_table(table, file_path, compression="zstd")

        written += len(sub)

    return written


batch_idx = 0
total_rows = 0

with INPUT_PATH.open("rb") as fh:
    while True:
        try:
            batch = pickle.load(fh)
        except EOFError:
            print("EOF")
            break
        except Exception as e:
            print(f"Stop on broken tail at batch_idx={batch_idx}: {e}")
            break

        if not isinstance(batch, list) or not batch:
            batch_idx += 1
            continue

        df = normalize_batch(batch)
        if df.empty:
            batch_idx += 1
            continue

        written = write_partitioned_files(df, batch_idx)
        total_rows += written

        print(
            f"batch={batch_idx} rows={len(df)} written={written} "
            f"min_dt={df['event_dt'].min()} max_dt={df['event_dt'].max()}"
        )
        batch_idx += 1

print("TOTAL_ROWS_WRITTEN:", total_rows)

batch=0 rows=10000 written=10000 min_dt=2026-07-20 21:38:38.220285889 max_dt=2026-07-20 21:38:54.472417236
batch=1 rows=10000 written=10000 min_dt=2026-07-20 21:38:54.493196045 max_dt=2026-07-20 21:39:04.818884033
batch=2 rows=10000 written=10000 min_dt=2026-07-20 21:39:04.826266113 max_dt=2026-07-20 21:39:15.617948975
batch=3 rows=10000 written=10000 min_dt=2026-07-20 21:39:15.637513916 max_dt=2026-07-20 21:39:27.549036865
batch=4 rows=10000 written=10000 min_dt=2026-07-20 21:39:27.568958984 max_dt=2026-07-20 21:39:38.670984131
batch=5 rows=10000 written=10000 min_dt=2026-07-20 21:39:38.687781006 max_dt=2026-07-20 21:39:50.825264160
batch=6 rows=10000 written=10000 min_dt=2026-07-20 21:39:50.846281982 max_dt=2026-07-20 21:40:00.637887939
batch=7 rows=10000 written=10000 min_dt=2026-07-20 21:40:00.649011963 max_dt=2026-07-20 21:40:09.233191895
batch=8 rows=10000 written=10000 min_dt=2026-07-20 21:40:09.244328125 max_dt=2026-07-20 21:40:18.836264893
batch=9 rows=10000 written=10000 min_

## Перевод паркет в дб и плот 3д+2д графиков 

In [ ]:


import duckdb
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.io as pio
from pathlib import Path
from plotly.offline import plot

pio.renderers.default = "browser"

PARQUET_GLOB = "output/spreads_parquet_v2_1/**/*.parquet"


def load_plot_df(
    start_dt: str,
    end_dt: str,
    spread_col: str = "spread_long",
    coin_offset: int = 0,
    top_n_coins: int = 30,
    max_rows_per_coin:int=500
) -> pd.DataFrame:
    con = duckdb.connect()

    extra_filters = []

    extra_filter_sql = ""
    if extra_filters:
        extra_filter_sql = " AND " + " AND ".join(extra_filters)

    query_sql = f"""
        WITH base_data AS (
            SELECT
                event_dt,
                base_coin,
                trigger,
                {spread_col} AS spread_value,
                okx_latency_ms,
                bybit_latency_ms,
                okx_freshness_ms,
                bybit_freshness_ms,
                max_latency_ms,
                max_freshness_ms
            FROM read_parquet('{PARQUET_GLOB}')
            WHERE event_dt >= TIMESTAMP '{start_dt}'
              AND event_dt < TIMESTAMP '{end_dt}'
              AND base_coin IS NOT NULL
              AND {spread_col} IS NOT NULL
        ),
        coin_counts AS (
            SELECT
                base_coin,
                COUNT(*) AS n_rows
            FROM base_data
            GROUP BY base_coin
        ),
        ranked_coins AS (
            SELECT
                base_coin,
                n_rows,
                ROW_NUMBER() OVER (ORDER BY n_rows DESC, base_coin ASC) AS coin_rank
            FROM coin_counts
        ),
        selected_coins AS (
            SELECT
                base_coin,
                coin_rank
            FROM ranked_coins
            WHERE coin_rank > {coin_offset}
              AND coin_rank <= {coin_offset + top_n_coins}
        )
        SELECT
            s.event_dt,
            s.base_coin,
            sc.coin_rank,
            s.trigger,
            s.spread_value,
            s.okx_latency_ms,
            s.bybit_latency_ms,
            s.okx_freshness_ms,
            s.bybit_freshness_ms,
            s.max_latency_ms,
            s.max_freshness_ms
        FROM base_data AS s
        INNER JOIN selected_coins AS sc
            ON s.base_coin = sc.base_coin
        WHERE 1=1
            {extra_filter_sql}
        ORDER BY sc.coin_rank, s.event_dt
    """

    df = con.execute(query_sql).df()

    if df.empty:
        return df

    df["event_dt"] = pd.to_datetime(df["event_dt"])
    df["coin_rank"] = df["coin_rank"].astype(int)
    df["legend_name"] = df["coin_rank"].astype(str) + ". " + df["base_coin"].astype(str)

    if max_rows_per_coin is not None:
        sampled_parts = []
        for _, sub in df.groupby("base_coin", sort=False):
            sub = sub.sort_values("event_dt").copy()
            if len(sub) > max_rows_per_coin:
                idx = np.linspace(0, len(sub) - 1, num=max_rows_per_coin, dtype=int)
                sub = sub.iloc[idx].copy()
            sampled_parts.append(sub)
        df = pd.concat(sampled_parts, ignore_index=True)

    df = df.sort_values(["coin_rank", "event_dt"]).reset_index(drop=True)
    return df

def inspect_plot_df(df: pd.DataFrame) -> None:
    if df.empty:
        print("plot_df is empty")
        return

    print("shape:", df.shape)
    print("time min:", df["event_dt"].min())
    print("time max:", df["event_dt"].max())
    print("duration sec:", (df["event_dt"].max() - df["event_dt"].min()).total_seconds())

    summary = (
        df.groupby("base_coin")
        .agg(
            n=("base_coin", "size"),
            tmin=("event_dt", "min"),
            tmax=("event_dt", "max"),
            spread_min=("spread_value", "min"),
            spread_max=("spread_value", "max"),
        )
        .sort_values("n", ascending=False)
        .head(20)
    )
    print(summary)

def plot_spread_2d(df: pd.DataFrame) -> None:
    if df.empty:
        raise ValueError("plot_df is empty")

    fig = go.Figure()

    for _, sub in df.groupby("coin_rank", sort=True):
        sub = sub.sort_values("event_dt")
        legend_name = sub["legend_name"].iloc[0]

        fig.add_trace(
            go.Scatter(
                x=sub["event_dt"],
                y=sub["spread_value"],
                mode="lines+markers",
                name=legend_name,
                line=dict(width=1.7),
                marker=dict(size=3),
                customdata=np.stack([
                    sub["trigger"].astype(str),
                    sub["okx_latency_ms"].fillna(np.nan),
                    sub["bybit_latency_ms"].fillna(np.nan),
                    sub["okx_freshness_ms"].fillna(np.nan),
                    sub["bybit_freshness_ms"].fillna(np.nan),
                ], axis=-1),
                hovertemplate=(
                    "coin=%{text}<br>"
                    "time=%{x}<br>"
                    "spread=%{y:.6f}<br>"
                    "trigger=%{customdata[0]}<br>"
                    "okx_latency_ms=%{customdata[1]:.2f}<br>"
                    "bybit_latency_ms=%{customdata[2]:.2f}<br>"
                    "okx_freshness_ms=%{customdata[3]:.2f}<br>"
                    "bybit_freshness_ms=%{customdata[4]:.2f}<extra></extra>"
                ),
                text=[legend_name] * len(sub),
            )
        )

    fig.update_layout(
        width=1400,
        height=700,
        xaxis_title="Event time",
        yaxis_title="Spread",
        hovermode="closest",
        legend=dict(font=dict(size=11)),
        margin=dict(l=50, r=20, t=60, b=40),
    )
    fig.show()
def plot_spread_3d(df: pd.DataFrame, x_unit: str = "seconds") -> None:
    if df.empty:
        raise ValueError("plot_df is empty")

    df = df.copy().sort_values(["coin_rank", "event_dt"])

    t0 = df["event_dt"].min()
    df["seconds_from_start"] = (df["event_dt"] - t0).dt.total_seconds().astype(float)
    df["hours_from_start"] = df["seconds_from_start"] / 3600.0

    if x_unit == "seconds":
        x_col = "seconds_from_start"
        x_title = "Seconds from window start"
    elif x_unit == "hours":
        x_col = "hours_from_start"
        x_title = "Hours from window start"
    else:
        raise ValueError("x_unit must be 'seconds' or 'hours'")

    print("X column:", x_col)
    print("X range:", df[x_col].min(), "->", df[x_col].max())
    print("event_dt range:", df["event_dt"].min(), "->", df["event_dt"].max())

    rank_min = df["coin_rank"].min()
    df["coin_y"] = df["coin_rank"] - rank_min

    y_tick_df = (
        df[["coin_rank", "coin_y", "legend_name"]]
        .drop_duplicates()
        .sort_values("coin_rank")
    )

    zmin = df["spread_value"].min()
    zmax = df["spread_value"].max()

    fig = go.Figure()

    for _, sub in df.groupby("coin_rank", sort=True):
        sub = sub.sort_values("event_dt")
        legend_name = sub["legend_name"].iloc[0]

        fig.add_trace(
            go.Scatter3d(
                x=sub[x_col].to_numpy(dtype=float),
                y=sub["coin_y"].to_numpy(dtype=float),
                z=sub["spread_value"].to_numpy(dtype=float),
                mode="lines+markers",
                name=legend_name,
                line=dict(width=4),
                marker=dict(size=2.8),
                opacity=0.9,
                customdata=np.stack([
                    sub["event_dt"].astype(str),
                    sub["trigger"].astype(str),
                    sub["okx_latency_ms"].fillna(np.nan).to_numpy(),
                    sub["bybit_latency_ms"].fillna(np.nan).to_numpy(),
                    sub["okx_freshness_ms"].fillna(np.nan).to_numpy(),
                    sub["bybit_freshness_ms"].fillna(np.nan).to_numpy(),
                ], axis=-1),
                hovertemplate=(
                    "coin=%{text}<br>"
                    "time=%{customdata[0]}<br>"
                    f"{x_title}=%{{x:.4f}}<br>"
                    "spread=%{z:.6f}<br>"
                    "trigger=%{customdata[1]}<br>"
                    "okx_latency_ms=%{customdata[2]:.2f}<br>"
                    "bybit_latency_ms=%{customdata[3]:.2f}<br>"
                    "okx_freshness_ms=%{customdata[4]:.2f}<br>"
                    "bybit_freshness_ms=%{customdata[5]:.2f}<extra></extra>"
                ),
                text=[legend_name] * len(sub),
            )
        )

    fig.update_layout(
        title="Spread map (3D)",
        width=1500,
        height=950,
        scene=dict(
            xaxis=dict(
                title=x_title,
                backgroundcolor="rgb(245,245,245)",
                gridcolor="lightgray",
                zerolinecolor="gray",
            ),
            yaxis=dict(
                title="Coin rank in block",
                tickmode="array",
                tickvals=y_tick_df["coin_y"].tolist(),
                ticktext=y_tick_df["legend_name"].tolist(),
                backgroundcolor="rgb(245,245,245)",
                gridcolor="lightgray",
            ),
            zaxis=dict(
                title="Spread",
                range=[zmin, zmax],
                backgroundcolor="rgb(245,245,245)",
                gridcolor="lightgray",
                zerolinecolor="black",
            ),
            aspectmode="manual",
            aspectratio=dict(x=2.6, y=1.4, z=1.1),
            camera=dict(eye=dict(x=1.8, y=1.7, z=1.15)),
        ),
        legend=dict(font=dict(size=11)),
        margin=dict(l=0, r=0, t=50, b=0),
    )
    fig.show()
def build_spread_2d_figure(df: pd.DataFrame) -> go.Figure:
    if df.empty:
        raise ValueError("plot_df is empty")

    fig = go.Figure()

    for _, sub in df.groupby("coin_rank", sort=True):
        sub = sub.sort_values("event_dt")
        legend_name = sub["legend_name"].iloc[0]

        fig.add_trace(
            go.Scatter(
                x=sub["event_dt"],
                y=sub["spread_value"],
                mode="lines+markers",
                name=legend_name,
                line=dict(width=1.7),
                marker=dict(size=3),
                customdata=np.stack([
                    sub["trigger"].astype(str),
                    sub["okx_latency_ms"].fillna(np.nan),
                    sub["bybit_latency_ms"].fillna(np.nan),
                    sub["okx_freshness_ms"].fillna(np.nan),
                    sub["bybit_freshness_ms"].fillna(np.nan),
                ], axis=-1),
                hovertemplate=(
                    "coin=%{text}<br>"
                    "time=%{x}<br>"
                    "spread=%{y:.6f}<br>"
                    "trigger=%{customdata[0]}<br>"
                    "okx_latency_ms=%{customdata[1]:.2f}<br>"
                    "bybit_latency_ms=%{customdata[2]:.2f}<br>"
                    "okx_freshness_ms=%{customdata[3]:.2f}<br>"
                    "bybit_freshness_ms=%{customdata[4]:.2f}<extra></extra>"
                ),
                text=[legend_name] * len(sub),
            )
        )

    fig.update_layout(
        title="2D spread",
        width=900,
        height=520,
        xaxis_title="Event time",
        yaxis_title="Spread",
        hovermode="closest",
        legend=dict(font=dict(size=10)),
        margin=dict(l=40, r=20, t=50, b=40),
    )
    return fig


def build_spread_3d_figure(df: pd.DataFrame, x_unit: str = "seconds") -> go.Figure:
    if df.empty:
        raise ValueError("plot_df is empty")

    df = df.copy().sort_values(["coin_rank", "event_dt"])

    t0 = df["event_dt"].min()
    df["seconds_from_start"] = (df["event_dt"] - t0).dt.total_seconds().astype(float)
    df["hours_from_start"] = df["seconds_from_start"] / 3600.0

    if x_unit == "seconds":
        x_col = "seconds_from_start"
        x_title = "Seconds from start"
    elif x_unit == "hours":
        x_col = "hours_from_start"
        x_title = "Hours from start"
    else:
        raise ValueError("x_unit must be 'seconds' or 'hours'")

    rank_min = df["coin_rank"].min()
    df["coin_y"] = df["coin_rank"] - rank_min

    y_tick_df = (
        df[["coin_rank", "coin_y", "legend_name"]]
        .drop_duplicates()
        .sort_values("coin_rank")
    )

    zmin = df["spread_value"].min()
    zmax = df["spread_value"].max()

    fig = go.Figure()

    for _, sub in df.groupby("coin_rank", sort=True):
        sub = sub.sort_values("event_dt")
        legend_name = sub["legend_name"].iloc[0]

        fig.add_trace(
            go.Scatter3d(
                x=sub[x_col].to_numpy(dtype=float),
                y=sub["coin_y"].to_numpy(dtype=float),
                z=sub["spread_value"].to_numpy(dtype=float),
                mode="lines+markers",
                name=legend_name,
                line=dict(width=4),
                marker=dict(size=2.5),
                opacity=0.9,
                customdata=np.stack([
                    sub["event_dt"].astype(str),
                    sub["trigger"].astype(str),
                    sub["okx_latency_ms"].fillna(np.nan).to_numpy(),
                    sub["bybit_latency_ms"].fillna(np.nan).to_numpy(),
                    sub["okx_freshness_ms"].fillna(np.nan).to_numpy(),
                    sub["bybit_freshness_ms"].fillna(np.nan).to_numpy(),
                ], axis=-1),
                hovertemplate=(
                    "coin=%{text}<br>"
                    "time=%{customdata[0]}<br>"
                    f"{x_title}=%{{x:.4f}}<br>"
                    "spread=%{z:.6f}<br>"
                    "trigger=%{customdata[1]}<br>"
                    "okx_latency_ms=%{customdata[2]:.2f}<br>"
                    "bybit_latency_ms=%{customdata[3]:.2f}<br>"
                    "okx_freshness_ms=%{customdata[4]:.2f}<br>"
                    "bybit_freshness_ms=%{customdata[5]:.2f}<extra></extra>"
                ),
                text=[legend_name] * len(sub),
            )
        )

    fig.update_layout(
        title="3D spread map",
        width=900,
        height=520,
        scene=dict(
            xaxis=dict(title=x_title),
            yaxis=dict(
                title="Coin rank",
                tickmode="array",
                tickvals=y_tick_df["coin_y"].tolist(),
                ticktext=y_tick_df["legend_name"].tolist(),
            ),
            zaxis=dict(
                title="Spread",
                range=[zmin, zmax],
            ),
            aspectmode="manual",
            aspectratio=dict(x=2.2, y=1.3, z=1.0),
            camera=dict(eye=dict(x=1.7, y=1.55, z=1.05)),
        ),
        legend=dict(font=dict(size=10)),
        margin=dict(l=0, r=0, t=50, b=0),
    )
    return fig

def build_dashboard_html(
    start_dt: str,
    end_dt: str,
    spread_col: str = "spread_long",
    block_size: int = 30,
    max_rows_per_coin: int = 200,
    x_unit: str = "seconds",
    output_file: str = "output/spread_dashboard_1.html",
    max_blocks= None,
):
    con = duckdb.connect()
    total_coins_sql = f"""
        SELECT COUNT(DISTINCT base_coin) AS n_coins
        FROM read_parquet('{PARQUET_GLOB}')
        WHERE event_dt >= TIMESTAMP '{start_dt}'
          AND event_dt < TIMESTAMP '{end_dt}'
          AND base_coin IS NOT NULL
          AND {spread_col} IS NOT NULL
    """
    total_coins = int(con.execute(total_coins_sql).fetchone()[0])

    if total_coins == 0:
        raise ValueError("No coins found in selected window")

    offsets = list(range(0, total_coins, block_size))
    if max_blocks is not None:
        offsets = offsets[:max_blocks]

    sections = []
    plotly_js_included = False

    for block_idx, offset in enumerate(offsets, start=1):
        df_block = load_plot_df(
            start_dt=start_dt,
            end_dt=end_dt,
            spread_col=spread_col,
            coin_offset=offset,
            top_n_coins=block_size,
            max_rows_per_coin=max_rows_per_coin,
        )

        if df_block.empty:
            continue

        tmin = df_block["event_dt"].min()
        tmax = df_block["event_dt"].max()
        dur_sec = (tmax - tmin).total_seconds()
        coin_count = df_block["base_coin"].nunique()
        row_count = len(df_block)

        fig3d = build_spread_3d_figure(df_block, x_unit=x_unit)
        fig2d = build_spread_2d_figure(df_block)

        div3d = plot(
            fig3d,
            include_plotlyjs=("cdn" if not plotly_js_included else False),
            output_type="div",
        )
        plotly_js_included = True

        div2d = plot(
            fig2d,
            include_plotlyjs=False,
            output_type="div",
        )

        section_html = f"""
        <section class="block">
            <div class="block-header">
                <h2>Block {block_idx}: coins {offset + 1}–{offset + coin_count}</h2>
                <div class="meta">
                    rows={row_count} | coins={coin_count} | time={tmin} → {tmax} | duration_sec={dur_sec:.2f}
                </div>
            </div>
            <div class="grid">
                <div class="panel left">
                    {div3d}
                </div>
                <div class="panel right">
                    {div2d}
                </div>
            </div>
        </section>
        """
        sections.append(section_html)
        print(f"built block={block_idx} offset={offset} rows={row_count} coins={coin_count}")

    html = f"""
    <!doctype html>
    <html lang="en">
    <head>
        <meta charset="utf-8">
        <title>Spread dashboard</title>
        <style>
            body {{
                margin: 0;
                font-family: Arial, sans-serif;
                background: #f5f5f5;
                color: #111;
            }}
            .page {{
                width: 100%;
                box-sizing: border-box;
                padding: 24px 20px 80px 20px;
            }}
            h1 {{
                margin: 0 0 8px 0;
                font-size: 28px;
            }}
            .sub {{
                margin-bottom: 24px;
                color: #555;
                font-size: 14px;
            }}
            .block {{
                background: white;
                border-radius: 12px;
                padding: 18px 18px 10px 18px;
                margin-bottom: 26px;
                box-shadow: 0 2px 10px rgba(0,0,0,0.06);
            }}
            .block-header {{
                margin-bottom: 12px;
            }}
            .block-header h2 {{
                margin: 0 0 6px 0;
                font-size: 20px;
            }}
            .meta {{
                font-size: 13px;
                color: #666;
            }}
            .grid {{
                display: grid;
                grid-template-columns: 1fr 1fr;
                gap: 18px;
                align-items: start;
            }}
            .panel {{
                min-width: 0;
                overflow: hidden;
            }}
            @media (max-width: 1400px) {{
                .grid {{
                    grid-template-columns: 1fr;
                }}
            }}
        </style>
    </head>
    <body>
        <div class="page">
            <h1>Spread dashboard</h1>
            <div class="sub">
                window={start_dt} → {end_dt} | spread_col={spread_col} | block_size={block_size} | max_rows_per_coin={max_rows_per_coin}
            </div>
            {''.join(sections)}
        </div>
    </body>
    </html>
    """

    output_path = Path(output_file)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    output_path.write_text(html, encoding="utf-8")
    print(f"saved html: {output_path.resolve()}")
    return output_path

dashboard_path = build_dashboard_html(
    start_dt="2026-07-16 22:00:00",
    end_dt="2026-07-17 08:00:00",
    spread_col="spread_short",
    block_size=5,
    max_rows_per_coin=1500,
    x_unit="seconds",
    output_file="output/spread_dashboard_1.html",
    max_blocks=2,   
)

print(dashboard_path)

built block=1 offset=0 rows=7500 coins=5
built block=2 offset=5 rows=7500 coins=5
saved html: /Users/mishatrubik/Desktop/spread/output/spread_dashboard_.html
output/spread_dashboard_.html


## Плот всех меток шорт и лонг спреда определенной монеты в фиксированном временном промежутке

In [ ]:
def load_coin_dual_spread_df(
    start_dt,
    end_dt,
    base_coin,
):
    con = duckdb.connect()

    query_sql = f"""
        SELECT
            event_dt,
            base_coin,
            trigger,
            spread_short,
            spread_long,
            okx_latency_ms,
            bybit_latency_ms,
            okx_freshness_ms,
            bybit_freshness_ms,
            max_latency_ms,
            max_freshness_ms
        FROM read_parquet('{PARQUET_GLOB}')
        WHERE event_dt >= TIMESTAMP '{start_dt}'
          AND event_dt < TIMESTAMP '{end_dt}'
          AND base_coin = '{base_coin}'
          AND (
                spread_short IS NOT NULL
                OR spread_long IS NOT NULL
              )
        ORDER BY event_dt
    """

    df = con.execute(query_sql).df()

    if df.empty:
        return df

    df["event_dt"] = pd.to_datetime(df["event_dt"])
    return df.reset_index(drop=True)

def inspect_coin_dual_spread_df(df):
    if df.empty:
        print("coin_df is empty")
        return

    print("shape:", df.shape)
    print("coin:", df["base_coin"].iloc[0])
    print("time min:", df["event_dt"].min())
    print("time max:", df["event_dt"].max())
    print("duration sec:", (df["event_dt"].max() - df["event_dt"].min()).total_seconds())

    print("spread_short non-null:", df["spread_short"].notna().sum())
    print("spread_long  non-null:", df["spread_long"].notna().sum())

    if df["spread_short"].notna().any():
        print("spread_short min:", df["spread_short"].min())
        print("spread_short max:", df["spread_short"].max())

    if df["spread_long"].notna().any():
        print("spread_long min:", df["spread_long"].min())
        print("spread_long max:", df["spread_long"].max())

    print("trigger counts:")
    print(df["trigger"].value_counts(dropna=False))

def plot_coin_short_long(df, title=None):
    if df.empty:
        raise ValueError("coin_df is empty")

    base_coin = df["base_coin"].iloc[0]
    df = df.sort_values("event_dt").copy()

    fig = go.Figure()

    fig.add_trace(
        go.Scatter(
            x=df["event_dt"],
            y=df["spread_short"],
            mode="lines+markers",
            name="spread_short",
            line=dict(width=1.6, color="#d62728"),
            marker=dict(size=3),
            customdata=np.stack([
                df["trigger"].astype(str),
                df["okx_latency_ms"].fillna(np.nan),
                df["bybit_latency_ms"].fillna(np.nan),
                df["okx_freshness_ms"].fillna(np.nan),
                df["bybit_freshness_ms"].fillna(np.nan),
                df["max_latency_ms"].fillna(np.nan),
                df["max_freshness_ms"].fillna(np.nan),
            ], axis=-1),
            hovertemplate=(
                f"coin={base_coin}<br>"
                "series=spread_short<br>"
                "time=%{x}<br>"
                "value=%{y:.6f}<br>"
                "trigger=%{customdata[0]}<br>"
                "okx_latency_ms=%{customdata[1]:.2f}<br>"
                "bybit_latency_ms=%{customdata[2]:.2f}<br>"
                "okx_freshness_ms=%{customdata[3]:.2f}<br>"
                "bybit_freshness_ms=%{customdata[4]:.2f}<br>"
                "max_latency_ms=%{customdata[5]:.2f}<br>"
                "max_freshness_ms=%{customdata[6]:.2f}<extra></extra>"
            ),
            connectgaps=False,
        )
    )

    fig.add_trace(
        go.Scatter(
            x=df["event_dt"],
            y=df["spread_long"],
            mode="lines+markers",
            name="spread_long",
            line=dict(width=1.6, color="#1f77b4"),
            marker=dict(size=3),
            customdata=np.stack([
                df["trigger"].astype(str),
                df["okx_latency_ms"].fillna(np.nan),
                df["bybit_latency_ms"].fillna(np.nan),
                df["okx_freshness_ms"].fillna(np.nan),
                df["bybit_freshness_ms"].fillna(np.nan),
                df["max_latency_ms"].fillna(np.nan),
                df["max_freshness_ms"].fillna(np.nan),
            ], axis=-1),
            hovertemplate=(
                f"coin={base_coin}<br>"
                "series=spread_long<br>"
                "time=%{x}<br>"
                "value=%{y:.6f}<br>"
                "trigger=%{customdata[0]}<br>"
                "okx_latency_ms=%{customdata[1]:.2f}<br>"
                "bybit_latency_ms=%{customdata[2]:.2f}<br>"
                "okx_freshness_ms=%{customdata[3]:.2f}<br>"
                "bybit_freshness_ms=%{customdata[4]:.2f}<br>"
                "max_latency_ms=%{customdata[5]:.2f}<br>"
                "max_freshness_ms=%{customdata[6]:.2f}<extra></extra>"
            ),
            connectgaps=False,
        )
    )

    fig.update_layout(
        title=title or f"{base_coin}: spread_short + spread_long",
        width=1500,
        height=750,
        xaxis_title="Event time",
        yaxis_title="Spread",
        hovermode="x unified",
        legend=dict(font=dict(size=11)),
        margin=dict(l=50, r=20, t=60, b=40),
    )
    fig.show()

coin_df = load_coin_dual_spread_df(
    start_dt="2026-07-16 22:00:00",
    end_dt="2026-07-17 2:00:00",
    base_coin="HOME",
)

inspect_coin_dual_spread_df(coin_df)
plot_coin_short_long(coin_df)

shape: (195350, 11)
coin: HOME
time min: 2026-07-16 22:00:00.039448975
time max: 2026-07-17 01:59:59.998001709
duration sec: 14399.958552
spread_short non-null: 195350
spread_long  non-null: 195350
spread_short min: -1.8724696356275243
spread_short max: 0.6539235412474758
spread_long min: -0.7901134521880059
spread_long max: 1.650755767700887
trigger counts:
trigger
bybit    116907
okx       78443
Name: count, dtype: int64
